In [3]:
import pandas as pd
import numpy as np
 
df = pd.read_csv("Team_Stats_Merge_2018_2025.csv")
target = df["made_playoffs"]
 
print("FEATURE SELECTION ANALYSIS")

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Target: made_playoffs (0={target.value_counts()[0]}, 1={target.value_counts()[1]})")

FEATURE SELECTION ANALYSIS
Dataset: 256 rows x 121 columns
Target: made_playoffs (0=148, 1=108)


### Outcome Leakage Variables

These variables directly encode win-loss outcomes. Playoff qualification is almost perfectly determined by wins, so including them would let the model memorize standings rather than learn structural performance patterns.

In [5]:
leakage_cols = ["tp_W", "tp_L", "tp_W-L%", "tp_PF", "tp_PA", "tp_PD", "tp_MoV"]

for col in leakage_cols:
    r = df[col].corr(target)
    desc = {
        "tp_W": "Wins (direct determinant of playoff qualification)",
        "tp_L": "Losses (complement of wins)",
        "tp_W-L%": "Win pct (derived from W and L)",
        "tp_PF": "Points For (raw scoring total)",
        "tp_PA": "Points Against (raw opponent scoring)",
        "tp_PD": "Point Differential (PF - PA)",
        "tp_MoV": "Margin of Victory (PD / games)",
    }
    print(f"{col:<13s} r = {r:+.3f} {desc[col]}")

tp_W          r = +0.805 Wins (direct determinant of playoff qualification)
tp_L          r = -0.797 Losses (complement of wins)
tp_W-L%       r = +0.804 Win pct (derived from W and L)
tp_PF         r = +0.623 Points For (raw scoring total)
tp_PA         r = -0.535 Points Against (raw opponent scoring)
tp_PD         r = +0.729 Point Differential (PF - PA)
tp_MoV        r = +0.730 Margin of Victory (PD / games)


The correlations are extremely high because these metrics directly determine the outcome we are predicting, so we are going to drop these 7 variables.

### Simple Rating System (SRS) Variables

SRS = Margin of Victory + Strength of Schedule, Since MoV is derived from point diferential, SRS inherits that leakage. OSRS and DSRS are components of SRS. tp_SoS measure schedule difficulty independently of team results, so this will be retained.

In [6]:
srs_cols = ["tp_SRS", "tp_OSRS", "tp_DSRS", "tp_SoS"]
print(f"{'Variable'} {'Corr w/ playoffs'} ")
for col in srs_cols:
    r = df[col].corr(target)
    print(f" {col:<10s} r = {r:+.3f}")

Variable Corr w/ playoffs 
 tp_SRS     r = +0.711
 tp_OSRS    r = +0.605
 tp_DSRS    r = +0.503
 tp_SoS     r = -0.256


Decision: DROP SRS, OSRS, DSRS. RETAIN tp_SoS

### Games Plaed

Determining whether to keep or drop the seven games-played columns, as there is extremely minor differences in games played with each season.

In [7]:
games_cols = ["off_g", "def_g", "off_pass_G", "off_rush_G", "def_pass_G"]
if "def_rush_G" in df.columns:
    games_cols.append("def_rush_G")
if "def_adv_G" in df.columns:
    games_cols.append("def_adv_G")
 
for col in games_cols:
    if col in df.columns:
        vals = df[col].value_counts()
        print(f"{col}:")
        for v, count in vals.items():
            print(f"{v}: {count} rows ({count/len(df)*100:.1f}%)")

off_g:
17: 158 rows (61.7%)
16: 98 rows (38.3%)
def_g:
17: 158 rows (61.7%)
16: 98 rows (38.3%)
off_pass_G:
17.0: 158 rows (61.7%)
16.0: 98 rows (38.3%)
off_rush_G:
17.0: 158 rows (61.7%)
16.0: 98 rows (38.3%)
def_pass_G:
17: 158 rows (61.7%)
16: 98 rows (38.3%)
def_rush_G:
17: 158 rows (61.7%)
16: 98 rows (38.3%)
def_adv_G:
17: 158 rows (61.7%)
16: 98 rows (38.3%)


Decision - Drop all as the values of 16 and 17 for all teams provide no discriminative signal for playoff prediction.

### Fourth Quarter Comebacks and Game Winning Drives

In [8]:
for col in ["off_pass_4QC", "off_pass_GWD"]:
    missing = df[col].isna().sum()
    pct = missing / len(df) * 100
    r = df[col].corr(target)
    print(f"{col}:")
    print(f"Missing: {missing} ({pct:.1f}%)")
    print(f"Corr w/ playoffs: r = {r:+.3f}")
    print(f"Missing by season:")
    for s in sorted(df["season"].unique()):
        n_miss = df[df["season"] == s][col].isna().sum()
        if n_miss > 0:
            print(f"{s}:{n_miss}")

off_pass_4QC:
Missing: 16 (6.2%)
Corr w/ playoffs: r = +0.257
Missing by season:
2019:2
2020:4
2021:5
2022:1
2023:1
2024:1
2025:2
off_pass_GWD:
Missing: 16 (6.2%)
Corr w/ playoffs: r = +0.343
Missing by season:
2019:2
2020:4
2021:5
2022:1
2023:1
2024:1
2025:2


Decision: Drropping both as they describe how wins happened, not why the team was competitive, 6.2% missing scattered across seasons, and misaligned with the efficiency-focused analytical framework.

### Tie's in the NFL tp_T

In [49]:
# Value distribution

print(df["tp_T"].value_counts(dropna=False).to_string())
# By Season
for s in sorted(df["season"].unique()):
    subset = df[df["season"] == s]
    nulls = subset["tp_T"].isna().sum()
    zeros = (subset["tp_T"] == 0).sum()
    ones = (subset["tp_T"] == 1).sum()
    print(f"{s}: null={nulls}, zero={zeros}, one={ones}")
 

tp_T
0.0    176
NaN     64
1.0     16
2018: null=0, zero=28, one=4
2019: null=0, zero=30, one=2
2020: null=0, zero=30, one=2
2021: null=0, zero=30, one=2
2022: null=0, zero=28, one=4
2023: null=32, zero=0, one=0
2024: null=32, zero=0, one=0
2025: null=0, zero=30, one=2


Decision: Drop — 176 zeros, 16 ones, 64 nulls in 2023-2024

### Multicollinearity Analysis 

Identifying all feature pairs with |r| > 0.90 to find clusters of near-identical variables that should be reduced

In [10]:
# Get all numeric columns (excluding keys and target)
numeric_cols = [c for c in df.select_dtypes(include=["int64", "float64"]).columns
                if c not in ["season", "made_playoffs"]]
 
corr_matrix = df[numeric_cols].corr()
 
# Find highly correlated pairs
high_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            c1, c2 = corr_matrix.columns[i], corr_matrix.columns[j]
            high_pairs.append((c1, c2, r))
 
high_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for c1, c2, r in high_pairs:
    print(f"  {c1:<20s}  {c2:<20s}  {r:+.3f}")
 
print(f"Total pairs with |r| > 0.90: {len(high_pairs)}")

  tp_PF                 off_pts               +1.000
  tp_PA                 def_pa                +1.000
  off_g                 def_g                 +1.000
  off_g                 off_pass_G            +1.000
  off_g                 off_rush_G            +1.000
  off_g                 def_pass_G            +1.000
  off_g                 def_rush_G            +1.000
  off_g                 def_adv_G             +1.000
  def_g                 off_pass_G            +1.000
  def_g                 off_rush_G            +1.000
  def_g                 def_pass_G            +1.000
  def_g                 def_rush_G            +1.000
  def_g                 def_adv_G             +1.000
  off_pass_G            off_rush_G            +1.000
  off_pass_G            def_pass_G            +1.000
  off_pass_G            def_rush_G            +1.000
  off_pass_G            def_adv_G             +1.000
  off_rush_G            def_pass_G            +1.000
  off_rush_G            def_rush_G            

Now, with these 83 pairs, we can further examine which features to drop.

I will pair these features by their cluster/category they pertain to, such as offensive passing, volume vs efficieny for offense, defensive passing, and rushing volume

In [11]:
 # Offensive passing cluster

off_pass_efficiency = ["off_pass_Y/A", "off_pass_AY/A", "off_pass_NY/A",
                       "off_pass_ANY/A", "off_pass_EXP", "off_pass_Rate"]
# these metrics are all nearly identical
eff_corr = df[off_pass_efficiency].corr()
print(eff_corr.round(3).to_string())
print()
 
# Loop through the 6 metrics computing how strongly each correlates with made_playoffs target variable
# using abs() to get the strength of the relationship
for col in off_pass_efficiency:
    r = abs(df[col].corr(target))
    print(f"  {col:<20s}  |r| = {r:.3f}")

                off_pass_Y/A  off_pass_AY/A  off_pass_NY/A  off_pass_ANY/A  off_pass_EXP  off_pass_Rate
off_pass_Y/A           1.000          0.923          0.940           0.882         0.812          0.831
off_pass_AY/A          0.923          1.000          0.906           0.972         0.891          0.963
off_pass_NY/A          0.940          0.906          1.000           0.943         0.909          0.836
off_pass_ANY/A         0.882          0.972          0.943           1.000         0.945          0.947
off_pass_EXP           0.812          0.891          0.909           0.945         1.000          0.889
off_pass_Rate          0.831          0.963          0.836           0.947         0.889          1.000

  off_pass_Y/A          |r| = 0.420
  off_pass_AY/A         |r| = 0.533
  off_pass_NY/A         |r| = 0.490
  off_pass_ANY/A        |r| = 0.563
  off_pass_EXP          |r| = 0.549
  off_pass_Rate         |r| = 0.560


Decision: Keep off_pass_ANY/A because it has the strongest relationsip with playoff qualification.
* you may notice off_pass_Rate is at 0.560, however I chose ANY/A because it also has the most comprehensive metric from a football perspective because it adjusts for sacks, interceptions, and touchdowns all in one number, while the others only adjust for some of those factors.

In [12]:
 # Volume vs efficiency for offense
off_volume = ["off_pass_Yds", "off_pass_Att", "off_pass_Cmp", "off_pass_TD",
              "off_pass_Lng", "off_pass_Y/C", "off_pass_Y/G", "off_pass_Yds.1"]

# confirming that the other offense metrics are safe to drop
for col in off_volume:
    if col in df.columns:
        r = abs(df[col].corr(target))
        print(f"{col:<20s}  |r| = {r:.3f}")

off_pass_Yds          |r| = 0.333
off_pass_Att          |r| = 0.021
off_pass_Cmp          |r| = 0.146
off_pass_TD           |r| = 0.487
off_pass_Lng          |r| = 0.053
off_pass_Y/C          |r| = 0.264
off_pass_Y/G          |r| = 0.312
off_pass_Yds.1        |r| = 0.414


off_pass_Cmp%, off_pass_TD%, off_pass_Int%, and off_pass_Rate are NOT included in this drop list even though they are passing metrics. These measure DISTINCT dimensions of passing performance (accuracy, scoring rate, mistake rate, and a composite rating) and are NOT highly correlated with each other or with off_pass_ANY/A at the |r| > 0.90 threshold. They each capture unique information, unlike the Y/A variants which are all slight mathematical adjustments of the same yards-per-attempt calculation. These are retained alongside ANY/A.

In [16]:
# Defensive passing cluster

def_pass_all = ["def_pass_Y/A", "def_pass_AY/A", "def_pass_NY/A",
                "def_pass_ANY/A", "def_pass_EXP", "def_pass_Rate"]

# same as offensive metircs above, these are all nearly identical as well
for col in def_pass_all:
    if col in df.columns:
        r = abs(df[col].corr(target))
        print(f"{col:<20s} |r| = {r:.3f}")

def_pass_Y/A         |r| = 0.389
def_pass_AY/A        |r| = 0.452
def_pass_NY/A        |r| = 0.408
def_pass_ANY/A       |r| = 0.452
def_pass_EXP         |r| = 0.415
def_pass_Rate        |r| = 0.437


Same logic as offensive passing: the Y/A variant cluster (Y/A, AY/A, NY/A, ANY/A, EXP, Rate) are near-perfect linear combinations of each other. I chose def_pass_ANY/A and dropped the rest.

def_pass_Cmp%, def_pass_TD%, def_pass_Int%, and def_pass_Rate are RETAINED because they measure distinct dimensions (completion accuracy allowed, scoring rate allowed, interception rate, and composite rating) that are not highly correlated with each other at |r| > 0.90. Additionally, defensive-specific metrics like def_pass_Sk, def_pass_QBHits, def_pass_TFL, and def_pass_Sk% are retained because they measure pass rush pressure, which has no direct equivalent in the offensive passing stats.

In [17]:
# Rushing Efficieny Metrics we are retaining

rush_efficiency = ["off_rush_Y/A", "def_rush_Y/A"]
for col in rush_efficiency:
    if col in df.columns:
        r = abs(df[col].corr(target))
        print(f"{col:<20s} |r| = {r:.3f}")

off_rush_Y/A         |r| = 0.151
def_rush_Y/A         |r| = 0.128


The above metrics measures yards per rush attempt, which normalizes how often a team runs. Similar to the ANY/A for passing.

In [20]:
# Rushing volume/redundant metrics dropped


rush_vol = ["off_rush_Lng", "off_rush_Y/G", "off_rush_Fmb", "def_rush_Y/G"]
rush_eff_corr = df[rush_vol].corr()
print(rush_eff_corr.round(3).to_string())

for col in rush_vol:
    if col in df.columns:
        r = abs(df[col].corr(target))
        print(f"{col:<20s}|r| = {r:.3f}")

              off_rush_Lng  off_rush_Y/G  off_rush_Fmb  def_rush_Y/G
off_rush_Lng         1.000         0.280        -0.019         0.004
off_rush_Y/G         0.280         1.000         0.095        -0.187
off_rush_Fmb        -0.019         0.095         1.000         0.049
def_rush_Y/G         0.004        -0.187         0.049         1.000
off_rush_Lng        |r| = 0.033
off_rush_Y/G        |r| = 0.322
off_rush_Fmb        |r| = 0.134
def_rush_Y/G        |r| = 0.409


In [23]:
# Show intercorrelation between Y/G and Y/A to further confirm redundancy

if "off_rush_Y/G" in df.columns and "off_rush_Y/A" in df.columns:
    r = df["off_rush_Y/G"].corr(df["off_rush_Y/A"])
    print(f"off_rush_Y/G vs off_rush_Y/A: r = {r:.3f}")
if "def_rush_Y/G" in df.columns and "def_rush_Y/A" in df.columns:
    r = df["def_rush_Y/G"].corr(df["def_rush_Y/A"])
    print(f"def_rush_Y/G vs def_rush_Y/A: r = {r:.3f}")

off_rush_Y/G vs off_rush_Y/A: r = 0.811
def_rush_Y/G vs def_rush_Y/A: r = 0.810


Decision: dropping the 4 metrics above because:
* off_rush_Lng is a single-play stat not a team indicator
* off_rush_Y/G is a volume stat not efficiency per attempt
* Fmb already captured in off_turnovers and off_fumbles
* def_rush_Y/G is also a volume stat not efficiency per attempt

### Defense Advance Duplicate Check

Comparing def_adv_ columns against existing def_pass_ columns to identify identical data stored under different names.

In [28]:
# identify all def_adv columns
all_adv_cols = [c for c in df.columns if c.startswith("def_adv_")]
len(all_adv_cols)

18

In [54]:
null_cols = []
complete_cols = []
 
for col in all_adv_cols:
    non_null = df[col].notna().sum()
    missing = df[col].isna().sum()
    pct = non_null / len(df) * 100
    if missing == len(df):
        null_cols.append(col)
    else:
        complete_cols.append(col)
    print(f"  {col:<16s}  {non_null:>4}  {missing:>4}  {pct:>6}%")


  def_adv_G          256     0   100.0%
  def_adv_Att        256     0   100.0%
  def_adv_Cmp        256     0   100.0%
  def_adv_Yds        256     0   100.0%
  def_adv_TD         256     0   100.0%
  def_adv_DADOT      256     0   100.0%
  def_adv_Air        256     0   100.0%
  def_adv_YAC        256     0   100.0%
  def_adv_Bltz       256     0   100.0%
  def_adv_Bltz%        0   256     0.0%
  def_adv_Hrry       256     0   100.0%
  def_adv_Hrry%        0   256     0.0%
  def_adv_QBKD       256     0   100.0%
  def_adv_QBKD%        0   256     0.0%
  def_adv_Sk         256     0   100.0%
  def_adv_Prss       256     0   100.0%
  def_adv_Prss%        0   256     0.0%
  def_adv_MTkl       256     0   100.0%


There are 4 columns without any data at all.

**Checking for duplicate columns**

Comparing the def_adv columns against the non adv defense columns to find exact or very close dupes

In [58]:
existing_cols = [c for c in df.select_dtypes(include=["int64", "float64"]).columns
                 if not c.startswith("def_adv_") and c not in ["season", "made_playoffs"]]

duplicate_cols = []
unique_cols = []

for adv_col in complete_cols:
    best_corr = 0
    best_match = ""
    is_exact = False

    for exist_col in existing_cols:
        r = abs(df[adv_col].corr(df[exist_col]))
        if r > best_corr:
            best_corr = r
            best_match = exist_col
            # Checking exact match only for very high correlations
            if r > 0.99:
                is_exact = (df[adv_col] == df[exist_col]).all()

    if best_corr >= 0.999 and is_exact:
        duplicate_cols.append(adv_col)
    elif best_corr >= 0.95:
        unique_cols.append(adv_col)  # keep for further analysis
    else:
        unique_cols.append(adv_col)
    print(f"{adv_col:<20s} {best_match:<20s} {best_corr:.4f}")

def_adv_G            off_g                1.0000
def_adv_Att          def_pass_Att         1.0000
def_adv_Cmp          def_pass_Cmp         1.0000
def_adv_Yds          def_pass_Yds         1.0000
def_adv_TD           def_pass_TD          1.0000
def_adv_DADOT        def_pass_Y/C         0.6259
def_adv_Air          def_pass_Y/G         0.7058
def_adv_YAC          def_pass_Cmp         0.6187
def_adv_Bltz         def_pass_Att         0.3624
def_adv_Hrry         off_g                0.3621
def_adv_QBKD         def_pass_QBHits      0.7952
def_adv_Sk           def_pass_Sk          1.0000
def_adv_Prss         def_pass_QBHits      0.7701
def_adv_MTkl         tp_PA                0.3652


Here we gather that there 6 duplicates to drop here with the corr = 1.0000, and we will keep the other unique columns for further evaluation.

After removing duplicates and nulls, there are 8 unique def_adv metrics remaining. Now we will evaluate each for predictive signalling and redundancy with existing features.

description of each advaned metric:
* def_adv_DADOT: Depth of target (how deep passes are thrown)
* def_adv_Air: Air yards allowed (pre-catch passing yards)
* def_adv_YAC: Yards after catch allowed (post-catch yards)
* def_adv_Bltz: Total blitzes called
* def_adv_Hrry: Hurries generated
* def_adv_QBKD: QB knockdowns
* def_adv_Prss: Total pressures (hurries + knockdowns + sacks)
* def_adv_MTkl: Missed tackles

In [61]:
unique_adv = ["def_adv_DADOT", "def_adv_Air", "def_adv_YAC", "def_adv_Bltz",
              "def_adv_Hrry", "def_adv_QBKD", "def_adv_Prss", "def_adv_MTkl"]
 
# Target correlations with playoffs
for col in unique_adv:
    r = df[col].corr(target)
    print(f"{col:<18s} r = {r:+.3f}")

def_adv_DADOT      r = +0.008
def_adv_Air        r = +0.342
def_adv_YAC        r = +0.295
def_adv_Bltz       r = +0.081
def_adv_Hrry       r = +0.125
def_adv_QBKD       r = +0.302
def_adv_Prss       r = +0.348
def_adv_MTkl       r = -0.220


Now checking these advanced stats to see if any of them are redundant with each other.

In [63]:
# Intercorrelation among unique advanced stats
print("--- Intercorrelation Among Unique Advanced Stats ---")
adv_corr = df[unique_adv].corr()
print(adv_corr.round(2).to_string())


--- Intercorrelation Among Unique Advanced Stats ---
               def_adv_DADOT  def_adv_Air  def_adv_YAC  def_adv_Bltz  def_adv_Hrry  def_adv_QBKD  def_adv_Prss  def_adv_MTkl
def_adv_DADOT           1.00         0.43        -0.28          0.01          0.09         -0.03          0.02         -0.10
def_adv_Air             0.43         1.00         0.32          0.18          0.19          0.20          0.22         -0.06
def_adv_YAC            -0.28         0.32         1.00          0.19          0.14          0.24          0.22          0.14
def_adv_Bltz            0.01         0.18         0.19          1.00          0.20          0.31          0.35          0.07
def_adv_Hrry            0.09         0.19         0.14          0.20          1.00          0.05          0.73          0.24
def_adv_QBKD           -0.03         0.20         0.24          0.31          0.05          1.00          0.63         -0.09
def_adv_Prss            0.02         0.22         0.22          0.35    

The correlation matrix above shows every pairs correlation and we can notice that def_adv_Hrry correlated with def_adv_Prss at r=0.73 telling us that keeping both may be somewhat redundant. If we can only keep one, pressures is the better choice because it's the broader metric that has to do with -- pressures = hurries + knockdowns + sacks combined.

In [ ]:
 # Cross-correlation with existing retained features

existing_def_features = ["def_pass_Sk", "def_pass_Sk%", "def_pass_QBHits",
                         "def_pass_TFL", "def_pass_ANY/A", "def_pass_Rate",
                         "def_pass_Cmp%", "def_score_pct", "def_yds_per_play",
                         "def_turnovers_forced"]
  
for adv_col in unique_adv:
    max_r = 0
    max_partner = ""
    for exist_col in existing_def_features:
        if exist_col in df.columns:
            r = abs(df[adv_col].corr(df[exist_col]))
            if r > max_r:
                max_r = r
                max_partner = exist_col
    print(f"  {adv_col:<18s}  {max_partner:<28s}  {max_r:.3f}")
 
 
# Final decisions

drop_adv = {
    "def_adv_QBKD": "Redundant with def_pass_QBHits (r=0.80) and def_adv_Prss (r=0.63)",
    "def_adv_Hrry": "Subsumed by def_adv_Prss (r=0.73) — Prss includes hurries",
    "def_adv_Bltz": "Weak signal (r=0.081 with playoffs) — blitz count ≠ blitz effectiveness",
    "def_adv_DADOT": "Near-zero signal (r=0.008) — depth of target not predictive of wins",
}
 
keep_adv = {
    "def_adv_Prss": "Best single pass rush metric (r=+0.348), broader than QBHits or Sk alone",
    "def_adv_Air": "Unique coverage signal (r=+0.342), no high correlation with existing features",
    "def_adv_YAC": "Unique tackling-after-catch signal (r=+0.295), complements Air yards",
    "def_adv_MTkl": "Unique tackling discipline signal (r=-0.220), not captured elsewhere",
}

  def_adv_DADOT       def_pass_Cmp%                 0.376
  def_adv_Air         def_yds_per_play              0.372
  def_adv_YAC         def_pass_Cmp%                 0.292
  def_adv_Bltz        def_pass_QBHits               0.306
  def_adv_Hrry        def_turnovers_forced          0.169
  def_adv_QBKD        def_pass_QBHits               0.795
  def_adv_Prss        def_pass_QBHits               0.770
  def_adv_MTkl        def_score_pct                 0.269
DROP (4):
  def_adv_QBKD          Redundant with def_pass_QBHits (r=0.80) and def_adv_Prss (r=0.63)
  def_adv_Hrry          Subsumed by def_adv_Prss (r=0.73) — Prss includes hurries
  def_adv_Bltz          Weak signal (r=0.081 with playoffs) — blitz count ≠ blitz effectiveness
  def_adv_DADOT         Near-zero signal (r=0.008) — depth of target not predictive of wins

RETAIN (4):
  def_adv_Prss          Best single pass rush metric (r=+0.348), broader than QBHits or Sk alone
  def_adv_Air           Unique coverage signal (r=+0.342

We can observe the correlation above. One thing to note is how def_adv_QBKD and def_pass_QBHits correlate at r=0.795. This means knockdowns and hits are measuring nearly the same thing, so keeping both add little new information. Looking at this on the other end of the spectrum we can note how def_adv_Air correlates at r=0.372 which means air yards capture something genuinely new that no existing feature already measures, so that's why for example we keep that adv metric instead and drop QBKD.

Missing Values Analysis

In [74]:
# Scanning entire dataset to find missing values
 
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
 
print(f"Total columns with missing values: {len(missing)}")
print(f"Overall completeness: {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")
print()
 
for col, count in missing.items():
    pct = count / len(df) * 100
    print(f"{col}:{count} missing ({pct:.1f}%)")
 
    # Show pattern by season
    if count < 256:
        print("By season:")
        for s in sorted(df["season"].unique()):
            n_miss = df[df["season"] == s][col].isna().sum()
            if n_miss > 0:
                print(f"{s}: {n_miss} missing out of 32")
    

Total columns with missing values: 9
Overall completeness: 96.38%

def_adv_Prss%:256 missing (100.0%)
def_adv_Hrry%:256 missing (100.0%)
def_adv_QBKD%:256 missing (100.0%)
def_adv_Bltz%:256 missing (100.0%)
tp_T:64 missing (25.0%)
By season:
2023: 32 missing out of 32
2024: 32 missing out of 32
off_pass_GWD:16 missing (6.2%)
By season:
2019: 2 missing out of 32
2020: 4 missing out of 32
2021: 5 missing out of 32
2022: 1 missing out of 32
2023: 1 missing out of 32
2024: 1 missing out of 32
2025: 2 missing out of 32
off_pass_4QC:16 missing (6.2%)
By season:
2019: 2 missing out of 32
2020: 4 missing out of 32
2021: 5 missing out of 32
2022: 1 missing out of 32
2023: 1 missing out of 32
2024: 1 missing out of 32
2025: 2 missing out of 32
def_pass_Int%:1 missing (0.4%)
By season:
2025: 1 missing out of 32
def_pass_Int:1 missing (0.4%)
By season:
2025: 1 missing out of 32
